In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRegressor
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score, mean_absolute_error, confusion_matrix, classification_report
import joblib


# Generate Dataset


In [50]:
# 1. قراءة الداتا الحقيقية
df_raw = pd.read_csv('exoplanet_dataset.csv')

# 2. بناء الداتا وتغيير أسماء الأعمدة عشان تمشي مع باقي الكود بتاعنا
df = pd.DataFrame()
df['radius'] = df_raw['pl_rade']
df['mass'] = df_raw['pl_bmasse']
df['flux'] = df_raw['st_teff']       # هنستخدم حرارة النجم كبديل للإشعاع
df['temp'] = df_raw['pl_eqt']
df['semi_major'] = df_raw['pl_orbsmax']
df['eccentricity'] = df_raw['pl_orbeccen']

# 3. تجهيز المخرجات (Target Variables)
# تحويل الـ ESI لنسبة مئوية
df['ESI_Score'] = df_raw['P_ESI'] * 100

# تحديد الكواكب الصالحة للحياة (العلماء بيعتبروا الكوكب صالح لو ESI أكبر من أو يساوي 80)
df['Habitability_Label'] = np.where(df['ESI_Score'] >= 80, 1, 0)

# 4. تنظيف سريع: مسح أي صفوف فيها بيانات ناقصة (NaN) عشان الموديل مايضربش إيرور
df = df.dropna()

print(f"Dataset Shape: {df.shape}")
print(f"Habitable Planets found (ESI >= 80): {df['Habitability_Label'].sum()}")
df.head()

Dataset Shape: (5326, 8)
Habitable Planets found (ESI >= 80): 25


,radius,mass,flux,temp,semi_major,eccentricity,ESI_Score,Habitability_Label
0,12.2,4914.898486,4874.0,1437.380,1.178,0.2380,8.764388,0
1,12.3,4684.814200,4213.0,607.766,1.530,0.0800,8.136597,0
2,13.1,1131.151301,4888.0,909.766,0.775,0.0000,7.742203,0
3,12.5,2828.672822,5338.0,830.966,2.839,0.3683,16.301977,0
4,13.5,565.737400,5750.0,606.000,1.660,0.6800,36.809290,0


# Data preprocessing

In [51]:
# Cell 3: Data Preprocessing with Stratified Split
# التأكد من عدم تسريب الـ ESI_Score في التدريب (Data Leakage Prevention)
X = df.drop(['Habitability_Label', 'ESI_Score'], axis=1)
y_class = df['Habitability_Label']
y_reg = df['ESI_Score']

# استخدام stratify=y_class لضمان توزيع الكواكب النادرة بالتساوي
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, stratify=y_class, random_state=42
)

# توحيد المقاييس
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

habitable_count = sum(y_class_train == 1)
k_neighbors = min(5, habitable_count - 1) if habitable_count > 1 else 1

if habitable_count > 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_class_resampled, y_class_train_resampled = smote.fit_resample(X_train_scaled, y_class_train)
else:
    X_train_class_resampled = X_train_scaled
    y_class_train_resampled = y_class_train

print(f"Original Habitable count in train: {habitable_count}")
print(f"Resampled Habitable count in train (After SMOTE): {sum(y_class_train_resampled == 1)}")

Original Habitable count in train: 20
Resampled Habitable count in train (After SMOTE): 4240


# Train & Evaluate Classification Model

In [52]:
# Cell 4: Train & Evaluate Classification Model (Preventing Overfitting)
# 1. إضافة قيود (Regularization) لمنع حفظ الداتا (Overfitting)
clf_model = RandomForestClassifier(
    n_estimators=150,
    class_weight='balanced',
    max_depth=7,               # تحديد عمق الشجرة عشان ماتحفظش التفاصيل الصغيرة
    min_samples_split=10,      # الشجرة مش هتتفرع إلا لو عندها 10 عينات على الأقل
    random_state=42
)
clf_model.fit(X_train_class_resampled, y_class_train_resampled)

# 2. التوقع على بيانات التدريب والاختبار للمقارنة
train_preds = clf_model.predict(X_train_class_resampled)
probs = clf_model.predict_proba(X_test_scaled)[:, 1]

# استخدام Threshold
custom_threshold = 0.4
test_preds = (probs >= custom_threshold).astype(int)

# 3. اختبار الـ Overfitting
train_acc = accuracy_score(y_class_train_resampled, train_preds) * 100
test_acc = accuracy_score(y_class_test, test_preds) * 100

print("=== Overfitting Check ===")
print(f"Training Accuracy: {train_acc:.2f}%")
print(f"Testing Accuracy:  {test_acc:.2f}%")
print("-" * 30)

# تقييم النموذج النهائي
print("=== Habitability Classifier Evaluation ===")
print("Confusion Matrix:")
print(confusion_matrix(y_class_test, test_preds))
print("\nClassification Report:")
print(classification_report(y_class_test, test_preds))

=== Overfitting Check ===
Training Accuracy: 99.88%
Testing Accuracy:  99.06%
------------------------------
=== Habitability Classifier Evaluation ===
Confusion Matrix:
[[1053    8]
 [   2    3]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1061
           1       0.27      0.60      0.38         5

    accuracy                           0.99      1066
   macro avg       0.64      0.80      0.69      1066
weighted avg       0.99      0.99      0.99      1066



# Train & Evaluate Regression Model

In [53]:
# Cell 5: Train & Evaluate Regression Model (With Sample Weights for High ESI)
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. السر هنا: إعطاء وزن مضاعف (عقاب أكبر) للكواكب اللي الـ ESI بتاعها عالي
# أي كوكب الـ ESI بتاعه فوق الـ 70، هنديه وزن 20 ضعف الكوكب العادي
weights = np.where(y_reg_train >= 70, 20, 1)

# 2. تعديل إعدادات الموديل ليكون أكثر مرونة في الوصول للأرقام العالية
reg_model = XGBRegressor(
    n_estimators=500,        # عدد أشجار أكبر
    learning_rate=0.05,
    max_depth=8,             # عمق أكبر عشان يقدر يلقط التفاصيل المعقدة للكواكب الصالحة
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

# 3. تدريب الموديل مع إجباره على التركيز على الكواكب ذات الوزن العالي
reg_model.fit(X_train_scaled, y_reg_train, sample_weight=weights)

# توقع النتائج
reg_preds = reg_model.predict(X_test_scaled)

# حساب مقاييس التقييم
rmse = np.sqrt(mean_squared_error(y_reg_test, reg_preds))
mae = mean_absolute_error(y_reg_test, reg_preds)
r2 = r2_score(y_reg_test, reg_preds)

print("=== Earth Similarity Score (Regression) Evaluation ===")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2 * 100:.2f}%")

=== Earth Similarity Score (Regression) Evaluation ===
RMSE: 4.58
MAE: 2.31
R2 Score: 87.00%


# Example for testing the Models(Sanity Check)

In [54]:
# Cell 5.5: Example Testing (Sanity Check) Before Saving

print("🔍 Testing Models with Unseen Examples from Test Data...\n")

# ==========================================
# 🌍 1. اختبار كوكب صالح للحياة (Habitable)
# ==========================================
# سحب أول كوكب صالح للحياة من بيانات الاختبار
hab_idx = y_class_test[y_class_test == 1].index[0]
hab_features = X_test.loc[hab_idx]

# معالجة الداتا (Scaler)
hab_scaled = scaler.transform(pd.DataFrame([hab_features]))

# توقعات الموديل
hab_class_pred = clf_model.predict(hab_scaled)[0]
hab_prob = clf_model.predict_proba(hab_scaled)[0].max() * 100
hab_esi_pred = max(0, min(100, reg_model.predict(hab_scaled)[0])) # التأكد إن الرقم بين 0 و 100

print("🌍 === Test Case 1: Habitable Planet ===")
print("📌 Input Features (Raw Data):")
print(hab_features.to_string())
print("-" * 30)
print("🎯 EXPECTED RESULT (True Label):")
print(f"- Habitability : Habitable (1)")
print(f"- ESI Score    : {y_reg_test.loc[hab_idx]:.2f} / 100")
print("-" * 30)
print("🤖 MODEL PREDICTION:")
pred_label_1 = 'Habitable' if hab_class_pred == 1 else 'Not Habitable'
print(f"- Habitability : {pred_label_1} (Confidence: {hab_prob:.1f}%)")
print(f"- ESI Score    : {hab_esi_pred:.2f} / 100")
print("=" * 45)

# ==========================================
# 🌑 2. اختبار كوكب غير صالح للحياة (Not Habitable)
# ==========================================
# سحب أول كوكب غير صالح للحياة من بيانات الاختبار
non_hab_idx = y_class_test[y_class_test == 0].index[0]
non_hab_features = X_test.loc[non_hab_idx]

# معالجة الداتا (Scaler)
non_hab_scaled = scaler.transform(pd.DataFrame([non_hab_features]))

# توقعات الموديل
non_hab_class_pred = clf_model.predict(non_hab_scaled)[0]
non_hab_prob = clf_model.predict_proba(non_hab_scaled)[0].max() * 100
non_hab_esi_pred = max(0, min(100, reg_model.predict(non_hab_scaled)[0]))

print("\n🌑 === Test Case 2: Not Habitable Planet ===")
print("🎯 EXPECTED RESULT (True Label):")
print(f"- Habitability : Not Habitable (0)")
print(f"- ESI Score    : {y_reg_test.loc[non_hab_idx]:.2f} / 100")
print("-" * 30)
print("🤖 MODEL PREDICTION:")
pred_label_2 = 'Habitable' if non_hab_class_pred == 1 else 'Not Habitable'
print(f"- Habitability : {pred_label_2} (Confidence: {non_hab_prob:.1f}%)")
print(f"- ESI Score    : {non_hab_esi_pred:.2f} / 100")
print("=" * 45)

🔍 Testing Models with Unseen Examples from Test Data...

🌍 === Test Case 1: Habitable Planet ===
📌 Input Features (Raw Data):
radius             1.0500
mass               1.1600
flux            3034.0000
temp             277.0000
semi_major         0.0259
eccentricity       0.0300
------------------------------
🎯 EXPECTED RESULT (True Label):
- Habitability : Habitable (1)
- ESI Score    : 96.84 / 100
------------------------------
🤖 MODEL PREDICTION:
- Habitability : Habitable (Confidence: 99.6%)
- ESI Score    : 89.25 / 100

🌑 === Test Case 2: Not Habitable Planet ===
🎯 EXPECTED RESULT (True Label):
- Habitability : Not Habitable (0)
- ESI Score    : 14.05 / 100
------------------------------
🤖 MODEL PREDICTION:
- Habitability : Not Habitable (Confidence: 100.0%)
- ESI Score    : 17.35 / 100


# Save Models & Scalers

In [55]:
# Cell 6: Save Models and Scaler
joblib.dump(clf_model, 'habitability_classifier.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

# الطريقة الصحيحة والآمنة لحفظ نموذج XGBoost
reg_model.save_model('earth_similarity_regressor.json')

print("✅ تم حفظ النماذج بنجاح!")

✅ تم حفظ النماذج بنجاح!
